# Medical Knowledge RAG

This notebook builds a retrieval-augmented generation (RAG) pipeline over real-world medical data sourced from Wikipedia.

It demonstrates:
- Fetching structured medical articles using the Wikipedia REST API
- Indexing documents with `minsearch`
- Querying the knowledge base using `GeneralizedRAG` (sync and agentic modes)
- Category-filtered search (e.g. restrict to cardiovascular conditions)

## 1 · Imports

In [4]:
import requests
from minsearch import Index
from openai import OpenAI
from generalized_rag import GeneralizedRAG

## 2 · Fetch medical articles from Wikipedia

We use the Wikipedia REST v1 `/page/summary/{title}` endpoint, which returns a plain-text extract (the introduction section) along with a short description for each article.

In [5]:
HEADERS = {"User-Agent": "medical-rag-demo/1.0 (educational project; contact: student)"}

def fetch_wikipedia_summary(title: str) -> dict:
    """Return title, description, and full intro-section extract for a Wikipedia page."""
    encoded = requests.utils.quote(title.replace(" ", "_"))
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{encoded}"
    resp = requests.get(url, headers=HEADERS, timeout=10)
    resp.raise_for_status()
    data = resp.json()
    return {
        "title": data.get("title", title),
        "description": data.get("description", ""),
        "content": data.get("extract", ""),
    }

In [6]:
# (condition Wikipedia title → medical category)
MEDICAL_CONDITIONS = {
    "Type 2 diabetes":                          "metabolic",
    "Hypertension":                             "cardiovascular",
    "Asthma":                                   "respiratory",
    "Pneumonia":                                "respiratory",
    "Myocardial infarction":                    "cardiovascular",
    "Major depressive disorder":                "mental health",
    "Alzheimer's disease":                      "neurological",
    "Stroke":                                   "neurological",
    "COVID-19":                                 "infectious",
    "Chronic obstructive pulmonary disease":    "respiratory",
    "Chronic kidney disease":                   "renal",
    "Osteoporosis":                             "musculoskeletal",
}

documents = []
for condition, category in MEDICAL_CONDITIONS.items():
    doc = fetch_wikipedia_summary(condition)
    doc["category"] = category
    documents.append(doc)
    print(f"  {doc['title']}  ({category})  — {len(doc['content'])} chars")

print(f"\nTotal documents: {len(documents)}")

  Type 2 diabetes  (metabolic)  — 748 chars
  Hypertension  (cardiovascular)  — 460 chars
  Asthma  (respiratory)  — 672 chars
  Pneumonia  (respiratory)  — 267 chars
  Myocardial infarction  (cardiovascular)  — 410 chars
  Major depressive disorder  (mental health)  — 609 chars
  Alzheimer's disease  (neurological)  — 698 chars
  Stroke  (neurological)  — 332 chars
  COVID-19  (infectious)  — 333 chars
  Chronic obstructive pulmonary disease  (respiratory)  — 481 chars
  Chronic kidney disease  (renal)  — 836 chars
  Osteoporosis  (musculoskeletal)  — 514 chars

Total documents: 12


In [7]:
# Inspect one document
sample = documents[0]
print("Title      :", sample["title"])
print("Category   :", sample["category"])
print("Description:", sample["description"])
print("\nContent (first 600 chars):")
print(sample["content"][:600])

Title      : Type 2 diabetes
Category   : metabolic
Description: Form of diabetes mellitus

Content (first 600 chars):
Type 2 diabetes (T2D), formerly known as adult-onset diabetes, is a form of diabetes mellitus that is characterized by high blood sugar, insulin resistance, and relative lack of insulin. Common symptoms include increased thirst, frequent urination, fatigue and unexplained weight loss. Other symptoms include increased hunger, having a sensation of pins and needles, and sores (wounds) that heal slowly. Symptoms often develop slowly. Long-term complications from high blood sugar include heart disease; stroke; diabetic retinopathy, which can result in blindness; kidney failure; and poor blood flow


## 3 · Build the search index

In [8]:
index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["category"],
)
index.fit(documents)
print("Index ready — documents:", len(documents))

Index ready — documents: 12


## 4 · Initialise GeneralizedRAG for the medical domain

In [9]:
MEDICAL_INSTRUCTIONS = """\
You are a knowledgeable medical information assistant.
Provide accurate, evidence-based answers drawn from the context supplied.

Always remind the user to consult a qualified healthcare professional for
personal medical advice, diagnosis, or treatment decisions.

If the answer cannot be found in the context, say:
"I don't have enough information to answer that reliably."
"""

openai_client = OpenAI()

medical_rag = GeneralizedRAG(
    index=index,
    llm_client=openai_client,
    domain="medical information",
    instructions=MEDICAL_INSTRUCTIONS,
    model="gpt-5.4-mini",
    boost_dict={"content": 3.0, "title": 1.5},
    num_results=5,
    context_fields=["title", "description", "content"],
)

## 5 · Synchronous RAG queries

In [10]:
q1 = "What are the main symptoms and warning signs of a heart attack?"
print("Q:", q1)
print("A:", medical_rag.rag(q1))

Q: What are the main symptoms and warning signs of a heart attack?
A: The main symptom of a heart attack is **retrosternal chest pain or discomfort**—pain or pressure in the center of the chest—that may **radiate to the left shoulder, arm, or jaw**. It can sometimes feel like **heartburn**.

**Warning signs include:**
- Chest pain or discomfort, especially if it spreads to the left shoulder, arm, or jaw
- A feeling like heartburn rather than typical chest pain

If you or someone else may be having a heart attack, **call emergency services immediately**. Please consult a qualified healthcare professional for personal medical advice, diagnosis, or treatment decisions.


In [11]:
q2 = "How is type 2 diabetes managed and what lifestyle changes are recommended?"
print("Q:", q2)
print("A:", medical_rag.rag(q2))

Q: How is type 2 diabetes managed and what lifestyle changes are recommended?
A: Type 2 diabetes is managed with a combination of **lifestyle changes, regular monitoring, and often medication** when needed. Because management depends on the person’s overall health and blood sugar levels, it’s important to work with a qualified healthcare professional for individualized advice.

**Lifestyle changes commonly recommended include:**
- **Healthy eating:** focusing on balanced meals and controlling carbohydrate intake
- **Regular physical activity:** exercise can help improve insulin sensitivity
- **Weight management:** losing excess weight can improve blood sugar control
- **Ongoing blood sugar monitoring:** to track how well the plan is working

Type 2 diabetes is a chronic condition characterized by high blood sugar and insulin resistance, and it can lead to serious long-term complications if not well controlled.

If you’d like, I can also help summarize common **medications** used for ty

In [12]:
q3 = "What is the difference between Alzheimer's disease and a stroke in terms of symptoms?"
print("Q:", q3)
print("A:", medical_rag.rag(q3))

Q: What is the difference between Alzheimer's disease and a stroke in terms of symptoms?
A: Alzheimer’s disease and stroke differ in how symptoms usually appear:

- **Alzheimer’s disease** typically causes a **gradual, progressive decline**. The most common early symptom is **difficulty remembering recent events**, and over time it can lead to **language problems, disorientation, mood changes, loss of motivation, self-neglect, behavioral issues**, and eventually loss of bodily functions.
- **Stroke** happens when blood flow to part of the brain is **reduced or blocked**, causing brain cells to die and that part of the brain to **stop functioning properly**. Symptoms are often **sudden** and depend on which brain area is affected.

If you want, I can also help compare them in a simple table. Please consult a qualified healthcare professional for personal medical advice, diagnosis, or treatment decisions.


In [13]:
medical_rag.cost()

Input tokens: 666


## 6 · Category-filtered queries

Pass a `filter_dict` at construction time to restrict search to a single medical category.

In [14]:
cardio_rag = GeneralizedRAG(
    index=index,
    llm_client=openai_client,
    domain="cardiovascular medicine",
    instructions=MEDICAL_INSTRUCTIONS,
    model="gpt-5.4-mini",
    boost_dict={"content": 3.0, "title": 1.5},
    filter_dict={"category": "cardiovascular"},
    num_results=5,
    context_fields=["title", "description", "content"],
)

q_cardio = "What are the shared risk factors between hypertension and myocardial infarction?"
print("Q:", q_cardio)
print("A:", cardio_rag.rag(q_cardio))

Q: What are the shared risk factors between hypertension and myocardial infarction?
A: Hypertension is a **major risk factor for myocardial infarction (heart attack)** because long-term high blood pressure increases the risk of **coronary artery disease**, which can reduce blood flow to the heart muscle and lead to MI.

Shared/related risk pathway from the context:
- **Hypertension**
- **Coronary artery disease** (a major complication/risk pathway linked to hypertension and a common cause of MI)

I don’t have enough information in the provided context to list additional shared risk factors reliably. Please consult a qualified healthcare professional for personal medical advice, diagnosis, or treatment decisions.


In [15]:
resp_rag = GeneralizedRAG(
    index=index,
    llm_client=openai_client,
    domain="respiratory medicine",
    instructions=MEDICAL_INSTRUCTIONS,
    model="gpt-5.4-mini",
    boost_dict={"content": 3.0, "title": 1.5},
    filter_dict={"category": "respiratory"},
    num_results=5,
    context_fields=["title", "description", "content"],
)

q_resp = "How do asthma and COPD differ in their causes and typical patient profiles?"
print("Q:", q_resp)
print("A:", resp_rag.rag(q_resp))

Q: How do asthma and COPD differ in their causes and typical patient profiles?
A: Asthma and COPD are both chronic lung diseases, but they differ in how they develop and who they typically affect:

- **Asthma:** A **long-term inflammatory disease of the airways** with **variable, recurring symptoms** such as wheezing, coughing, chest tightness, and shortness of breath. Symptoms can flare when triggered by **allergens, pollen, dust, or other inhaled particles**, and may worsen at night or with exercise. It often appears as episodes or attacks rather than steadily worsening breathing problems.

- **COPD:** A **progressive lung disease** that develops when the **airways and lung tissue become damaged**, causing long-term poor airflow and reduced oxygen exchange. It **develops gradually over time** and tends to cause steadily worsening breathing difficulty.

**Typical patient profile:**  
The context does not give detailed age or risk-group profiles, but it does indicate that **asthma is m

## 7 · Agentic RAG

Complex multi-hop questions benefit from the agentic mode, where the model can issue multiple search calls with different keywords before composing a final answer.

In [16]:
q_agentic = (
    "Compare the long-term complications of type 2 diabetes and hypertension. "
    "Which organs are affected and what preventive measures overlap?"
)

print("Q:", q_agentic)
print()
agentic_answer = medical_rag.agenticRAG(q_agentic)
print("A:", agentic_answer)

Q: Compare the long-term complications of type 2 diabetes and hypertension. Which organs are affected and what preventive measures overlap?

Search iterations: 3
A: Type 2 diabetes and hypertension both cause **long-term damage to multiple organs**, and they **overlap heavily in cardiovascular and kidney complications**.

### Organs affected

**Type 2 diabetes**
- **Heart and blood vessels**: heart disease, stroke
- **Kidneys**: kidney failure
- **Eyes**: diabetic retinopathy, which can lead to blindness
- **Lower limbs / peripheral circulation**: poor blood flow, possible amputations
- Also can affect **nerves** and wound healing indirectly, though the provided context specifically mentions pins-and-needles and slow-healing sores

**Hypertension**
- **Heart and blood vessels**: stroke, coronary artery disease, heart failure, atrial fibrillation, peripheral arterial disease
- **Kidneys**: chronic kidney disease
- **Eyes**: vision loss
- **Brain**: dementia

### Main overlap
Both condit

In [17]:
q_neuro = (
    "What are the similarities and differences in how Alzheimer's disease "
    "and major depressive disorder affect memory and daily functioning?"
)

print("Q:", q_neuro)
print()
neuro_answer = medical_rag.agenticRAG(q_neuro)
print("A:", neuro_answer)

Q: What are the similarities and differences in how Alzheimer's disease and major depressive disorder affect memory and daily functioning?

Search iterations: 3
A: Alzheimer’s disease and major depressive disorder can both affect memory and daily life, but they do so in different ways.

Similarities:
- Both can cause problems with concentration and remembering information.
- Both can interfere with daily functioning, such as work, social life, and self-care.
- Both may lead to withdrawal from family or usual activities.

Differences:
- Alzheimer’s disease is a progressive neurodegenerative disorder. Its early hallmark is difficulty remembering recent events, and over time it also causes language problems, disorientation, self-neglect, and broader decline in daily functioning.
- Major depressive disorder is primarily a mood disorder. It is characterized by persistent low mood, loss of interest or pleasure, and low self-esteem. Memory problems in depression are usually related to poor co

## 8 · Token-cost comparison: sync vs agentic

In [18]:
comparison_q = "What preventive measures reduce the risk of stroke?"

# Sync
sync_answer = medical_rag.rag(comparison_q)
print("=== Sync RAG ===")
print(sync_answer)
print()
medical_rag.cost()

=== Sync RAG ===
The context does not provide specific stroke-prevention measures, but it does state that **hypertension (high blood pressure) is a major risk factor for stroke**. So, **controlling blood pressure** is an important way to reduce stroke risk.

I don’t have enough information to answer that reliably beyond that. Please consult a qualified healthcare professional for personal medical advice, diagnosis, or treatment decisions.

Input tokens: 609


In [19]:
# Agentic (resets loop_count each call)
print("=== Agentic RAG ===")
agentic_stroke = medical_rag.agenticRAG(comparison_q)
print(agentic_stroke)

=== Agentic RAG ===
Search iterations: 2
Preventive measures that reduce the risk of stroke include:

- **Control high blood pressure** — this is one of the most important stroke risk factors.
- **Manage diabetes** — keep blood sugar under control.
- **Lower cholesterol and treat other cardiovascular risk factors** as advised by a clinician.
- **Do not smoke** and avoid secondhand smoke.
- **Be physically active** regularly.
- **Eat a heart-healthy diet** and maintain a healthy weight.

If you have **atrial fibrillation** or other heart conditions, treating them appropriately can also help reduce stroke risk.

I don’t have enough information to give more specific prevention advice reliably. Please consult a qualified healthcare professional for personal medical advice, diagnosis, or treatment decisions.
